In [ ]:
# Use local repository code (no pip install)


In [ ]:

import warnings
warnings.filterwarnings('ignore')
from pysmatch import Matcher, load_demo
import pandas as pd
import numpy as np

In [ ]:
data = load_demo()
data

In [ ]:
test_data_full = data[data.loan_status == "Default"].copy() # Use .copy() to avoid SettingWithCopyWarning
control_data_full = data[data.loan_status == "Fully Paid"].copy()
treatment_var = 'is_default' 
exclude_cols = ['loan_status']

In [ ]:
np.random.seed(20250604)
matcher_instance = Matcher(test=test_data_full,
                           control=control_data_full,
                           yvar=treatment_var, # Matcher will create 'is_default' column internally
                           exclude=exclude_cols)

In [ ]:
# ============ (1) Noraml train (Without optuna) =============
# m.fit_scores(balance=True, nmodels=10, n_jobs=3, model_type='knn')
# m.fit_scores(balance=True, nmodels=10, n_jobs=3, model_type='tree', max_iter=100)
matcher_instance.fit_scores(balance=True, nmodels=10, model_type='linear', n_jobs=2)
# ============ (2) Utilize optuna (Only train one best model) =============
# m.fit_scores(
#     balance=True,
#     model_type='tree',
#     max_iter=200,
#     use_optuna=True,
#     n_trials=15
# )
print(f"Number of models fitted: {len(matcher_instance.models)}")
print(f"Average model accuracy: {np.mean(matcher_instance.model_accuracy):.4f}")

In [ ]:
matcher_instance.predict_scores()
print(matcher_instance.data[['record_id', matcher_instance.treatment_col, 'scores']].head())


In [ ]:
matcher_instance.plot_scores()

In [ ]:
matcher_instance.tune_threshold(method='min', nmatches=1, rng=np.arange(0.0001, 0.0051, 0.0005))

In [ ]:
matcher_instance.match(threshold=0.001, nmatches=1, method='min', replacement=True, exhaustive_matching=False)
display_cols_standard = ['record_id', matcher_instance.treatment_col, 'scores']
print(matcher_instance.matched_data[display_cols_standard].head())
matcher_instance.plot_matched_scores()
freq_df = matcher_instance.record_frequency()
matcher_instance.assign_weight_vector()
print("top 6 matched data")
print(matcher_instance.matched_data.sort_values("match_id").head(6))

In [ ]:
matcher_instance.match(threshold=0.001, nmatches=1, exhaustive_matching=True)
display_cols_exhaustive = ['record_id', matcher_instance.treatment_col, 'scores', 'match_id', 'matched_as', 'pair_score_diff']
actual_display_cols_exhaustive = [col for col in display_cols_exhaustive if col in matcher_instance.matched_data.columns]
print(matcher_instance.matched_data[actual_display_cols_exhaustive].head())
matcher_instance.plot_matched_scores()

In [ ]:
control_usage_freq = matcher_instance.record_frequency()
print(control_usage_freq.head())
matcher_instance.assign_weight_vector()
weight_display_cols = ['record_id', 'match_id', 'weight']
if 'matched_as' in matcher_instance.matched_data.columns:
    weight_display_cols.append('matched_as')
actual_weight_display_cols = [col for col in weight_display_cols if col in matcher_instance.matched_data.columns]
print(matcher_instance.matched_data[actual_weight_display_cols].head())

In [ ]:
continuous_comparison = matcher_instance.compare_continuous(return_table=True, plot_result=True)
print(continuous_comparison)

In [ ]:
grade_prop_test = matcher_instance.prop_test('grade')
print(grade_prop_test)

In [ ]:
categorical_comparison = matcher_instance.compare_categorical(return_table=True, plot_result=True)
print(categorical_comparison)